In [ ]:
!pip install langchain langchain-openai langchain-core

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from pprint import pprint

In [ ]:
# 1. Knowledge Base (Documents)
docs = [
    "LangChain is a framework for developing LLM applications.",
    "RAG fetches relevant external context to minimize hallucinations.",
    "Fine-tuning adapts model weights on domain-specific datasets.",
]

In [ ]:
# API KEY SETUP (Hardcoded for demo)
# -------------------------------------------------------------
MY_OPENAI_API_KEY = ""

In [ ]:
# 2. Embeddings & Vector Store (Hiding the Math)
print("Creating Vector Database...")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small",api_key=MY_OPENAI_API_KEY)
vectorstore = InMemoryVectorStore.from_texts(docs, embedding=embeddings)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

In [ ]:
query = "What helps reduce LLM hallucinations?"
retrieved_docs = retriever.invoke(query)
context_text = retrieved_docs[0].page_content
print(f"\n[Retrieved Context]: {context_text}")

In [ ]:
# augmented prompt template
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer the user's question based ONLY on the following context:\n\n{context}"),
    ("user", "{question}")
])

In [ ]:
print(type(prompt_template))

In [ ]:
formatted_messages = prompt_template.format_messages(
    context=context_text,
    question=query
)

In [ ]:
pprint(formatted_messages)

In [ ]:
print(prompt_template.messages[0].prompt.template)

In [ ]:
pprint(formatted_messages)

In [ ]:
print(type(formatted_messages))

In [ ]:
print(type(formatted_messages[0].content))

In [ ]:
print("\nGenerating Answer...")

llm = ChatOpenAI(
    model="gpt-4o-mini", 
    api_key=MY_OPENAI_API_KEY
)

response = llm.invoke(formatted_messages)

print(f"\n[Final Answer]: {response.content}")